In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc qiskit-ibm-transpiler
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Passes de transpilation assistées par IA

Les passes de transpilation assistées par IA sont des passes qui remplacent directement les passes Qiskit « traditionnelles » pour certaines tâches de transpilation. Elles produisent souvent de meilleurs résultats que les algorithmes heuristiques existants (comme une profondeur et un nombre de CNOT plus faibles), tout en étant beaucoup plus rapides que des algorithmes d'optimisation tels que les solveurs de satisfaisabilité booléenne. Les passes de transpilation IA s'exécutent dans ton environnement local.

> **Note:** Les passes de transpilation assistées par IA sont en version bêta et sont susceptibles d'évoluer.
> Si tu as des commentaires ou si tu veux contacter l'équipe de développement, utilise ce [canal Qiskit Slack Workspace](https://qiskit.slack.com/archives/C06KF8YHUAU).

Les passes suivantes sont actuellement disponibles :

**Passes de routage**
 - `AIRouting` : Sélection du layout et routage de circuit

**Passes de synthèse de circuit**
 - `AICliffordSynthesis` : Synthèse de circuit Clifford
 - `AILinearFunctionSynthesis` : Synthèse de circuit à fonction linéaire
 - `AIPermutationSynthesis` : Synthèse de circuit de permutation

Pour utiliser les passes de transpilation IA, installe d'abord le package `qiskit-ibm-transpiler`. Consulte la [documentation API de qiskit-ibm-transpiler](https://docs.quantum.ibm.com/api/qiskit-ibm-transpiler) pour en savoir plus sur les différentes options disponibles.

## Exécuter les passes de transpilation IA localement ou dans le cloud
Commence par installer `qiskit-ibm-transpiler` avec quelques dépendances supplémentaires comme suit :

In [1]:
from qiskit.transpiler import PassManager
from qiskit.circuit.library import efficient_su2
from qiskit_ibm_transpiler.ai.routing import AIRouting
from qiskit_ibm_runtime import QiskitRuntimeService
import logging

backend = QiskitRuntimeService().backend("ibm_fez")
ai_passmanager = PassManager(
    [
        AIRouting(
            backend=backend,
            optimization_level=2,
            layout_mode="optimize",
            local_mode=True,
        )
    ]
)


circuit = efficient_su2(101, entanglement="circular", reps=1)
logging.getLogger(
    "qiskit_ibm_transpiler.wrappers.ai_local_synthesis"
).setLevel(logging.WARNING)
transpiled_circuit = ai_passmanager.run(circuit)

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Après installation des dépendances supplémentaires, le mode par défaut d'exécution des passes de transpilation assistées par IA consiste à utiliser ta machine locale.

## Passe de routage IA
La passe `AIRouting` joue à la fois le rôle d'étape de layout et d'étape de routage. Elle peut être utilisée au sein d'un `PassManager` comme suit :

In [ ]:
from qiskit.transpiler import PassManager

from qiskit_ibm_transpiler.ai.routing import AIRouting
from qiskit_ibm_transpiler.ai.synthesis import AILinearFunctionSynthesis
from qiskit_ibm_transpiler.ai.collection import CollectLinearFunctions
from qiskit.circuit.library import efficient_su2

ibm_kingston = QiskitRuntimeService().backend("ibm_kingston")
ai_passmanager = PassManager(
    [
        AIRouting(
            backend=ibm_kingston,
            optimization_level=3,
            layout_mode="optimize",
            local_mode=True,
        ),  # Route circuit
        CollectLinearFunctions(),  # Collect Linear Function blocks
        AILinearFunctionSynthesis(
            backend=ibm_kingston, local_mode=True
        ),  # Re-synthesize Linear Function blocks
    ]
)

circuit = efficient_su2(10, entanglement="full", reps=1)

transpiled_circuit = ai_passmanager.run(circuit)

Fetching 127 files:   0%|          | 0/127 [00:00<?, ?it/s]

The synthesis respects the coupling map of the device: it can be run safely after other routing passes without disturbing the circuit, so the overall circuit will still follow the device restrictions. By default, the synthesis will replace the original sub-circuit only if the synthesized sub-circuit improves the original (currently only checking CNOT count), but this can be forced to always replace the circuit by setting `replace_only_if_better=False`.

The following synthesis passes are available from `qiskit_ibm_transpiler.ai.synthesis`:

- *AICliffordSynthesis*: Synthesis for [Clifford](/docs/api/qiskit/qiskit.quantum_info.Clifford) circuits (blocks of `H`, `S`, and `CX` gates). Currently up to nine qubit blocks.
- *AILinearFunctionSynthesis*: Synthesis for [Linear Function](/docs/api/qiskit/qiskit.circuit.library.LinearFunction) circuits (blocks of `CX` and `SWAP` gates). Currently up to nine qubit blocks.
- *AIPermutationSynthesis*: Synthesis for [Permutation](/docs/api/qiskit/qiskit.circuit.library.Permutation#permutation) circuits (blocks of `SWAP` gates). Currently available for 65, 33, and 27 qubit blocks.
- *AIPauliNetworkSynthesis*: Synthesis for Pauli Network circuits (blocks of `H`, `S`, `SX`, `CX`, `RX`, `RY` and `RZ` gates). Currently up to six qubit blocks.

We expect to gradually increase the size of the supported blocks.

All passes use a thread pool to send several requests in parallel. By default, the number for max threads is the number of cores plus four (default values for the `ThreadPoolExecutor` Python object). However, you can set your own value with the `max_threads` argument at pass instantiation. For example, the following line instantiates the `AILinearFunctionSynthesis` pass, which allows it to use a maximum of 20 threads.

```python
AILinearFunctionSynthesis(backend=ibm_torino, max_threads=20)  # Re-synthesize Linear Function blocks using 20 threads max
```

You can also set the environment variable `AI_TRANSPILER_MAX_THREADS` to the desired number of maximum threads, and all synthesis passes instantiated after that will use that value.

For the AI synthesis passes to synthesize a sub-circuit, it must lay on a connected subgraph of the coupling map (one way to do this is with a routing pass before collecting the blocks, but this is not the only way to do it). The synthesis passes will automatically check that the specific subgraph is supported, and if not, it will raise a warning and leave the original sub-circuit unchanged.

The following custom collection passes for Cliffords, Linear Functions and Permutations that can be imported from `qiskit_ibm_transpiler.ai.collection` also complement the synthesis passes:

- *CollectCliffords*: Collects Clifford blocks as `Instruction` objects and stores the original sub-circuit to compare against it after synthesis.
- *CollectLinearFunctions*: Collects blocks of `SWAP` and `CX` as `LinearFunction` objects and stores the original sub-circuit to compare against it after synthesis.
- *CollectPermutations*: Collects blocks of `SWAP` circuits as `Permutations`.
- *CollectPauliNetworks*: Collects Pauli Network blocks and stores the original sub-circuit to compare against it after synthesis.

These custom collection passes limit the sizes of the collected sub-circuits so they are supported by the AI-powered synthesis passes. Therefore, it is recommended to use them after the routing passes and before the synthesis passes for a better overall optimization.

## Hybrid heuristic-AI circuit transpilation

The `qiskit-ibm-transpiler` allows you to configure a hybrid pass manager that combines the best of Qiskit's heuristic and the AI-powered transpiler passes. This feature behaves similarly to the Qiskit `generate_pass_manager` method. A typical way to use `generate_ai_pass_manager` is as follows:

In [ ]:
from qiskit_ibm_transpiler import generate_ai_pass_manager
from qiskit.circuit.library import efficient_su2
from qiskit_ibm_runtime import QiskitRuntimeService


backend = QiskitRuntimeService().backend("ibm_kingston")
kingston_coupling_map = backend.coupling_map


su2_circuit = efficient_su2(101, entanglement="circular", reps=1)

ai_transpiler_pass_manager = generate_ai_pass_manager(
    coupling_map=kingston_coupling_map,
    ai_optimization_level=3,
    optimization_level=3,
    ai_layout_mode="optimize",
)

ai_su2_transpiled_circuit = ai_transpiler_pass_manager.run(su2_circuit)

Ici, le `backend` détermine la coupling map sur laquelle effectuer le routage, le `optimization_level` (1, 2 ou 3) détermine l'effort de calcul à consacrer au processus (une valeur plus élevée donne généralement de meilleurs résultats mais prend plus de temps), et le `layout_mode` précise comment gérer la sélection du layout.
Le `layout_mode` comprend les options suivantes :

- `keep` : Respecte le layout défini par les passes de transpilation précédentes (ou utilise le layout trivial si aucun n'est défini). Il est généralement utilisé uniquement lorsque le circuit doit s'exécuter sur des qubits spécifiques de l'appareil. Il produit souvent de moins bons résultats car il offre moins de marge pour l'optimisation.
- `improve` : Utilise le layout défini par les passes de transpilation précédentes comme point de départ. C'est utile lorsque tu as une bonne estimation initiale du layout ; par exemple, pour des circuits construits de manière à suivre approximativement la coupling map de l'appareil. C'est aussi utile si tu veux essayer d'autres passes de layout spécifiques combinées avec la passe `AIRouting`.
- `optimize` : Il s'agit du mode par défaut. Il fonctionne mieux pour les circuits généraux pour lesquels tu n'as peut-être pas de bonnes estimations de layout. Ce mode ignore les sélections de layout précédentes.
- `local_mode` : Ce drapeau détermine où s'exécute la passe `AIRouting`. Si `False`, `AIRouting` s'exécute à distance via le Qiskit Transpiler Service. Si `True`, le package tente d'exécuter la passe dans ton environnement local, avec un repli vers le mode cloud si les dépendances requises ne sont pas trouvées.

## Passes de synthèse de circuit IA
Les passes de synthèse de circuit IA te permettent d'optimiser des portions de différents types de circuit ([Clifford](https://docs.quantum.ibm.com/api/qiskit/qiskit.quantum_info.Clifford), [Linear Function](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.LinearFunction), [Permutation](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.Permutation#permutation), Pauli Network) en les re-synthétisant. Une façon typique d'utiliser la passe de synthèse est la suivante :